In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

###**Setup run once**


In [0]:
#spark.sql("CREATE TABLE IF NOT EXISTS databricks_cata.bronze.patient_flow")

###**Azure Event Hub Configuration**

In [0]:
event_hub_namespace = "hospital-analytics-stream.servicebus.windows.net"
event_hub_name = "hospital-analytics-stream"
event_hub_conn_str = dbutils.secrets.get(
    scope="hospitalanalyticsvaultscope", key="eventhub-connection"
)

kafka_options = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}:9093",
    "subscribe": event_hub_name,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="$ConnectionString" password="{event_hub_conn_str}";'
    ),
    "startingOffsets": "earliest",
    "failOnDataLoss": "false",
}

###**Read from eventhub**

In [0]:
raw_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

# Cast the payload to a JSON string
json_df = raw_df.selectExpr("CAST(value AS STRING) AS raw_json")

###**Write Stream to Bronze Table**

In [0]:
# Target table and checkpoint
bronze_patient_flow = "databricks_cata.bronze.patient_flow"
bronze_path = "abfss://bronze@databrickssp.dfs.core.windows.net/patient_flow_uc"
checkpoint_path = "abfss://bronze@databrickssp.dfs.core.windows.net/_checkpoints/patient_flow_bronze"

# Write to the Unity Catalog table
query = (
    json_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("path", bronze_path)
    .trigger(availableNow=True)
    .toTable(bronze_patient_flow)
)

query.awaitTermination()

###**Fake Records to test SCD Type 2**

In [0]:

from pyspark.sql.functions import *
import json

manual_json = {
    "patient_id": "e0827fc1-8ee4-49e0-b92a-6962404daf7e",
    "gender": "Male",
    "age": 16,
    "department": "Surgery",
    "admission_time": "2026-11-26T10:39:03.560739",
    "discharge_time": "2026-11-24T23:25:03.560739",
    "bed_id": 199,
    "hospital_id": 5,
}

bronze_patient_flow = "databricks_cata.bronze.patient_flow"

manual_df = spark.createDataFrame([(json.dumps(manual_json),)], ["raw_json"])

(
    manual_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_patient_flow)
)

In [0]:
display(spark.read.table("databricks_cata.bronze.patient_flow"))